In [11]:
import os
import json
import cv2
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# --- CONFIGURATION ---
# Adjust these paths to match your actual Kaggle input path
DATA_ROOT = "/kaggle/input/license-plate-datset/dataset"  # Or wherever you uploaded it
TRAIN_ROOT = os.path.join(DATA_ROOT, "train")

# Image parameters
IMG_HEIGHT = 64  # Standard height for OCR models
IMG_WIDTH = 256  # We will resize plates to this
BATCH_SIZE = 8

class LRLPRDataset(Dataset):
    def __init__(self, root_dir, transform=None, mode="train"):
        self.root_dir = root_dir
        self.transform = transform
        self.mode = mode
        
        # 1. Recursively find all 'track_XXXXX' folders
        self.tracks = glob.glob(os.path.join(root_dir, "**", "track_*"), recursive=True)
        self.tracks = sorted(self.tracks)
        
        print(f"[{mode.upper()}] Found {len(self.tracks)} tracks in {root_dir}")

    def __len__(self):
        return len(self.tracks)

    def _get_image_path(self, track_path, prefix, frame_num):
        """Robustly finds an image file (png, jpg, jpeg) for a given frame."""
        filename_base = f"{prefix}-{frame_num:03d}"
        for ext in [".png", ".jpg", ".jpeg", ".JPG"]:
            full_path = os.path.join(track_path, filename_base + ext)
            if os.path.exists(full_path):
                return full_path
        raise FileNotFoundError(f"Image {filename_base} not found in {track_path}")

    def _get_label_from_json(self, data):
        """Helper to find the text label regardless of the key name."""
        # List of potential keys the dataset might use
        potential_keys = ['text', 'license_plate', 'plate', 'label', 'plate_text']
        
        for key in potential_keys:
            if key in data:
                return data[key]
        
        # If we can't find it, print available keys to help debug
        raise KeyError(f"Could not find text label. Available keys: {list(data.keys())}")

    def __getitem__(self, idx):
        track_path = self.tracks[idx]
        
        # 2. Load the 5 Low-Resolution (LR) Images
        lr_images = []
        try:
            for i in range(1, 6): 
                img_path = self._get_image_path(track_path, "lr", i)
                image = Image.open(img_path).convert("RGB")
                if self.transform:
                    image = self.transform(image)
                lr_images.append(image)
        except Exception as e:
            print(f"Error loading images in: {track_path}")
            raise e
        
        lr_tensor = torch.stack(lr_images)
        
        # 3. Load Labels (Only for training)
        label = ""
        if self.mode == "train":
            json_path = os.path.join(track_path, "annotations.json")
            try:
                with open(json_path, 'r') as f:
                    data = json.load(f)
                    # Use our helper to find the right key
                    label = self._get_label_from_json(data)
            except Exception as e:
                print(f"Error parsing JSON in {track_path}: {e}")
                # Return a dummy label so training doesn't crash on one bad file
                label = "ERROR" 
        
        return lr_tensor, label, track_path

hui


In [4]:
import json
import os

# Let's peek at the first track's annotation
json_path = "/kaggle/input/license-plate-datset/dataset/train/Scenario-A/Brazilian/track_00001/annotations.json"

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
        print("✅ Keys found in JSON:", list(data.keys()))
        print("Example Content:", data)
else:
    print("❌ Could not find the specific file to test. Running generic search...")
    # Find any json file
    files = glob.glob("/kaggle/input/license-plate-datset/dataset/**/*.json", recursive=True)
    if files:
        with open(files[0], 'r') as f:
            print(f"Checking file: {files[0]}")
            print("Keys found:", list(json.load(f).keys()))

✅ Keys found in JSON: ['plate_layout', 'plate_text', 'corners']
Example Content: {'plate_layout': 'Brazilian', 'plate_text': 'AVL5215', 'corners': {'lr-001.png': {'top-left': [2, 2], 'top-right': [28, 2], 'bottom-right': [28, 14], 'bottom-left': [2, 13]}, 'lr-002.png': {'top-left': [2, 2], 'top-right': [30, 2], 'bottom-right': [30, 14], 'bottom-left': [2, 14]}, 'lr-003.png': {'top-left': [2, 2], 'top-right': [30, 3], 'bottom-right': [30, 16], 'bottom-left': [2, 15]}, 'lr-004.png': {'top-left': [2, 2], 'top-right': [33, 3], 'bottom-right': [33, 17], 'bottom-left': [2, 15]}, 'lr-005.png': {'top-left': [2, 2], 'top-right': [33, 4], 'bottom-right': [33, 17], 'bottom-left': [2, 16]}, 'hr-001.png': {'top-left': [2, 2], 'top-right': [57, 4], 'bottom-right': [57, 29], 'bottom-left': [2, 27]}, 'hr-002.png': {'top-left': [2, 2], 'top-right': [58, 3], 'bottom-right': [58, 28], 'bottom-left': [3, 27]}, 'hr-003.png': {'top-left': [2, 2], 'top-right': [55, 4], 'bottom-right': [55, 27], 'bottom-left'

In [7]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# --- 1. THE TOKENIZER (Handling Text <-> Numbers) ---
class Tokenizer:
    def __init__(self):
        # All possible characters in Brazilian/Mercosur plates
        # A-Z, 0-9. The hyphen '-' is usually ignored in recognition or treated as blank.
        self.chars = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        self.char_map = {char: i + 1 for i, char in enumerate(self.chars)} # 1-based index
        self.idx_map = {i + 1: char for i, char in enumerate(self.chars)}
        self.blank_idx = 0 # CTC requires a "blank" token at index 0
        
    def text_to_labels(self, text):
        # Cleans text and converts to tensor
        text = text.upper().replace("-", "") # Remove hyphens to standardise
        return torch.LongTensor([self.char_map[c] for c in text if c in self.char_map])

    def labels_to_text(self, labels):
        # Converts tensor back to text (decoding CTC)
        res = []
        for i in range(len(labels)):
            idx = labels[i].item()
            # CTC decoding: collapse repeats and ignore blanks
            if idx != self.blank_idx:
                if i == 0 or idx != labels[i-1].item():
                    res.append(self.idx_map.get(idx, ""))
        return "".join(res)
        
    def decode_greedy(self, logits):
        # Takes model output (Logits) -> Text
        # logits shape: (Time, Batch, Classes)
        pred_indices = torch.argmax(logits, dim=2).permute(1, 0) # (Batch, Time)
        results = []
        for i in range(pred_indices.size(0)):
            raw_pred = pred_indices[i]
            # Simple greedy decode
            decoded = []
            prev_idx = -1
            for idx in raw_pred:
                idx = idx.item()
                if idx != self.blank_idx and idx != prev_idx:
                    decoded.append(self.idx_map.get(idx, ""))
                prev_idx = idx
            results.append("".join(decoded))
        return results

# --- 2. THE CRNN MODEL (FIXED) ---
class CRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size=256):
        super(CRNN, self).__init__()
        
        # A. CNN Backbone (ResNet18)
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) # Updated syntax
        self.cnn = nn.Sequential(*list(resnet.children())[:-2]) 
        
        # Adaptation layer: 512 channels -> 256 channels
        self.conv_adapt = nn.Conv2d(512, 256, kernel_size=1) 

        # B. RNN (Bidirectional LSTM)
        # FIXED: input_size is 512 because (256 channels * 2 pixels height) = 512
        self.rnn = nn.LSTM(input_size=512, hidden_size=hidden_size, 
                           num_layers=2, bidirectional=True, batch_first=True)
        
        # C. Output Head
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x):
        # 1. Extract Features
        features = self.cnn(x) # -> (Batch, 512, H/32, W/32)
        features = self.conv_adapt(features) # -> (Batch, 256, 2, 8)
        
        # 2. Prepare for RNN
        b, c, h, w = features.size()
        features = features.permute(0, 3, 1, 2) # (Batch, Width, Channels, Height)
        features = features.reshape(b, w, -1)   # (Batch, Width, Channels*Height) -> (Batch, 8, 512)
        
        # 3. Sequence Modeling
        rnn_out, _ = self.rnn(features) 
        
        # 4. Prediction
        logits = self.fc(rnn_out) 
        
        # Permute for CTC Loss (Time, Batch, Vocab)
        return logits.permute(1, 0, 2)

# --- VERIFY FIX ---
tokenizer = Tokenizer()
model = CRNN(vocab_size=len(tokenizer.chars) + 1)
dummy_input = torch.randn(8, 3, 64, 256)
out = model(dummy_input)
print(f"✅ Fixed! Output shape: {out.shape}")

✅ Fixed! Output shape: torch.Size([8, 8, 37])


In [13]:
# --- RE-INITIALIZE DATASET & LOADER ---
# This forces the notebook to use the NEW class definition you just wrote
train_dataset = LRLPRDataset(root_dir=TRAIN_ROOT, transform=data_transforms, mode="train")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print("✅ Dataset and Loader refreshed with the new Robust Class!")

[TRAIN] Found 20000 tracks in /kaggle/input/license-plate-datset/dataset/train
✅ Dataset and Loader refreshed with the new Robust Class!


In [14]:
import torch.optim as optim
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 10
LEARNING_RATE = 1e-3

print(f"🚀 Training on device: {DEVICE}")

# 1. Setup Model, Loss, Optimizer
model = model.to(DEVICE)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 2. Training Loop
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    correct_tracks = 0
    total_tracks = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch_idx, (images_seq, labels, _) in enumerate(progress_bar):
        # images_seq shape: (Batch, 5, 3, 64, 256)
        # We merge Batch and 5-Frames dimensions to train on EVERY frame
        # New shape: (Batch * 5, 3, 64, 256)
        batch_size, seq_len, c, h, w = images_seq.shape
        images = images_seq.view(-1, c, h, w).to(DEVICE)
        
        # Prepare Targets
        # We need to repeat the labels 5 times because we have 5 images per label
        targets = []
        target_lengths = []
        for label in labels:
            encoded = tokenizer.text_to_labels(label)
            targets.extend([encoded] * 5) # Repeat label for all 5 frames
            target_lengths.extend([len(encoded)] * 5)
            
        targets = torch.cat(targets).to(DEVICE)
        target_lengths = torch.tensor(target_lengths).to(DEVICE)
        
        # Forward Pass
        optimizer.zero_grad()
        preds = model(images) # (Time, Batch*5, Vocab)
        
        # CTC Loss Requirements
        input_lengths = torch.full(size=(preds.size(1),), fill_value=preds.size(0), dtype=torch.long).to(DEVICE)
        
        loss = criterion(preds, targets, input_lengths, target_lengths)
        
        # Backward Pass
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # --- QUICK ACCURACY CHECK (Greedy Decode) ---
        if batch_idx % 100 == 0:
            # Decode just the first image of the batch to show progress
            decoded_text = tokenizer.decode_greedy(preds[:, 0:1, :])[0]
            true_text = labels[0]
            progress_bar.set_postfix({"Loss": loss.item(), "Pred": decoded_text, "True": true_text})

    print(f"Epoch {epoch+1} Average Loss: {epoch_loss / len(train_loader):.4f}")

    # Save Checkpoint
    torch.save(model.state_dict(), f"crnn_epoch_{epoch+1}.pth")
    print(f"💾 Saved checkpoint: crnn_epoch_{epoch+1}.pth")

🚀 Training on device: cuda


Epoch 1/10:   0%|          | 0/2500 [00:00<?, ?it/s]

Epoch 1 Average Loss: 2.8806
💾 Saved checkpoint: crnn_epoch_1.pth


Epoch 2/10:   0%|          | 0/2500 [00:00<?, ?it/s]

Epoch 2 Average Loss: 2.4567
💾 Saved checkpoint: crnn_epoch_2.pth


...

Epoch 8 Average Loss: 0.9050
💾 Saved checkpoint: crnn_epoch_8.pth


Epoch 9/10:   0%|          | 0/2500 [00:00<?, ?it/s]

Epoch 9 Average Loss: 0.8736
💾 Saved checkpoint: crnn_epoch_9.pth


Epoch 10/10:   0%|          | 0/2500 [00:00<?, ?it/s]

Epoch 10 Average Loss: 0.8525
💾 Saved checkpoint: crnn_epoch_10.pth


In [15]:
def decode_with_confidence(logits, tokenizer):
    """
    Decodes logits into text AND calculates the average confidence.
    Args:
        logits: (Time, Batch, Vocab)
        tokenizer: Your Tokenizer instance
    Returns:
        List of tuples: [("ABC1234", 0.98), ("XYZ9876", 0.85), ...]
    """
    # 1. Apply Softmax to get probabilities (0.0 to 1.0)
    probs = F.softmax(logits, dim=2) # (Time, Batch, Vocab)
    
    # Get max probability and index for each time step
    max_probs, pred_indices = torch.max(probs, dim=2)
    
    # Transpose to (Batch, Time) for easy iteration
    pred_indices = pred_indices.permute(1, 0)
    max_probs = max_probs.permute(1, 0)
    
    results = []
    
    for b in range(pred_indices.size(0)):
        raw_indices = pred_indices[b]
        raw_scores = max_probs[b]
        
        decoded_chars = []
        char_scores = []
        
        prev_idx = -1
        
        for t in range(len(raw_indices)):
            idx = raw_indices[t].item()
            score = raw_scores[t].item()
            
            # CTC Logic: Only add if not blank and not repeated
            if idx != tokenizer.blank_idx and idx != prev_idx:
                decoded_chars.append(tokenizer.idx_map.get(idx, ""))
                char_scores.append(score)
            
            prev_idx = idx
            
        final_text = "".join(decoded_chars)
        
        # Confidence is the average probability of the characters
        if len(char_scores) > 0:
            final_conf = sum(char_scores) / len(char_scores)
        else:
            final_conf = 0.0
            
        results.append((final_text, final_conf))
        
    return results

In [16]:
# --- CONFIGURATION ---
TEST_ROOT = os.path.join(DATA_ROOT, "test-public")
MODEL_PATH = "crnn_epoch_10.pth" # Change this to your best saved epoch
OUTPUT_FILE = "predictions.txt"

# 1. Load Test Data
# Note: mode="test" means it won't look for annotations.json
test_dataset = LRLPRDataset(root_dir=TEST_ROOT, transform=data_transforms, mode="test")
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"📂 Loaded {len(test_dataset)} test tracks.")

# 2. Load Model
model = CRNN(vocab_size=len(tokenizer.chars) + 1).to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print(f"✅ Loaded weights from {MODEL_PATH}")
else:
    print(f"⚠️ Warning: Model file {MODEL_PATH} not found. Make sure training finished!")

model.eval()

# 3. Generate Predictions
predictions = []

print("🚀 Starting Inference...")
with torch.no_grad():
    for images_seq, _, paths in tqdm(test_loader):
        # Flatten batch (Batch, 5, C, H, W) -> (Batch*5, C, H, W)
        b, seq, c, h, w = images_seq.shape
        images = images_seq.view(-1, c, h, w).to(DEVICE)
        
        # Forward Pass
        logits = model(images) # (Time, Batch*5, Vocab)
        
        # Decode
        results = decode_with_confidence(logits, tokenizer)
        
        # Aggregate results (we have 5 predictions per track)
        # Strategy: Pick the one with the HIGHEST confidence
        for i in range(b):
            track_results = results[i*5 : (i+1)*5] # Get the 5 results for this track
            
            # Sort by confidence (descending)
            track_results.sort(key=lambda x: x[1], reverse=True)
            
            best_text, best_conf = track_results[0]
            track_id = os.path.basename(paths[i]) # e.g., "track_00001"
            
            predictions.append(f"{track_id}, {best_text};{best_conf:.4f}")

# 4. Save to File
with open(OUTPUT_FILE, "w") as f:
    for line in predictions:
        f.write(line + "\n")

print(f"✅ Done! Saved {len(predictions)} predictions to {OUTPUT_FILE}")

[TEST] Found 1000 tracks in /kaggle/input/license-plate-datset/dataset/test-public
📂 Loaded 1000 test tracks.
✅ Loaded weights from crnn_epoch_10.pth
🚀 Starting Inference...


  0%|          | 0/125 [00:00<?, ?it/s]

✅ Done! Saved 1000 predictions to predictions.txt
